# Data Prep: Daily Hires + Bank Holidays + Calendar/Lag Features

Loads the raw TfL daily cycle hire counts and the England bank holiday
dates, joins them, engineers calendar and lag/rolling features, and
writes the result to `data/processed/daily_hires_with_holidays.csv`.

In [1]:
from pathlib import Path

import pandas as pd

raw_dir = Path("..") / "data" / "raw"
processed_dir = Path("..") / "data" / "processed"

## Load raw data

In [2]:
hires = pd.read_csv(raw_dir / "tfl_daily_cycle_hires.csv", parse_dates=["date"])
holidays_raw = pd.read_csv(raw_dir / "england_public_holidays.csv")

hires.shape, holidays_raw.shape

((5877, 2), (141, 4))

## Tidy the holidays table

In [3]:
holidays = holidays_raw.copy()
holidays["date"] = pd.to_datetime(holidays["Formatted Date"], format="%d %B %Y")
holidays = holidays.rename(columns={"Bank Holiday": "bank_holiday_name"})[
    ["date", "bank_holiday_name"]
]

assert holidays["date"].is_unique, "expected one row per holiday date"
holidays.head()

,date,bank_holiday_name
0,2010-01-01,New Year's Day
1,2010-04-02,Good Friday
2,2010-04-05,Easter Monday
3,2010-05-03,Early May Bank Holiday
4,2010-05-31,Spring Bank Holiday


## Join holidays onto the daily hires

In [4]:
# Left join - every hire date is kept, holiday info is only present on the
# days that are actually bank holidays
df = hires.merge(holidays, on="date", how="left")
df["is_bank_holiday"] = df["bank_holiday_name"].notna()

assert len(df) == len(hires), "join must not add or drop hire rows"
df.rename(columns={"number_of_bicycle_hires": "cycle_hire_count"}, inplace=True)
df["is_bank_holiday"].value_counts()

is_bank_holiday
False    5743
True      134
Name: count, dtype: int64

## Calendar features

In [5]:
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df.head()

,date,cycle_hire_count,bank_holiday_name,is_bank_holiday,day_of_week,month,year
0,2010-07-30,6897,NaN,False,4,7,2010
1,2010-07-31,5564,NaN,False,5,7,2010
2,2010-08-01,4303,NaN,False,6,8,2010
3,2010-08-02,6642,NaN,False,0,8,2010
4,2010-08-03,7966,NaN,False,1,8,2010


## Lag and rolling features

All lag/rolling values use `.shift(1)` before rolling so a row never sees
its own day's count. The first 27 rows will have `NaN` in `rolling_mean_28`
until enough history has accumulated - handled downstream (see the
training notebook) rather than dropped here, so the processed file keeps
one row per raw hire date.

In [6]:
df["lag_1"] = df["cycle_hire_count"].shift(1)
df["lag_7"] = df["cycle_hire_count"].shift(7)
df["lag_28"] = df["cycle_hire_count"].shift(28)
df["rolling_mean_7"] = df["cycle_hire_count"].shift(1).rolling(7).mean()
df["rolling_mean_28"] = df["cycle_hire_count"].shift(1).rolling(28).mean()
df.tail()

,date,cycle_hire_count,bank_holiday_name,is_bank_holiday,day_of_week,month,year,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
5872,2026-08-27,25233,NaN,False,3,8,2026,31336.0,29955.0,33850.0,28253.571429,29107.821429
5873,2026-08-28,25781,NaN,False,4,8,2026,25233.0,26985.0,30382.0,27579.000000,28800.071429
5874,2026-08-29,15779,NaN,False,5,8,2026,25781.0,26356.0,27306.0,27407.000000,28635.750000
5875,2026-08-30,20335,NaN,False,6,8,2026,15779.0,23128.0,25503.0,25896.000000,28224.071429
5876,2026-08-31,19426,Summer Bank Holiday,True,0,8,2026,20335.0,28125.0,29800.0,25497.000000,28039.500000


## Save the processed dataset

In [7]:
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "daily_hires_with_holidays.csv"
df.to_csv(output_path, index=False)
output_path

WindowsPath('../data/processed/daily_hires_with_holidays.csv')